In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()  # 여기서 새 kaggle.json 선택
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle (1).json


In [ ]:
!kaggle datasets download -d pauldoun/celeb-df-v2 -p /content/
!unzip -q /content/celeb-df-v2.zip -d /content/celebdf_v2

Dataset URL: https://www.kaggle.com/datasets/pauldoun/celeb-df-v2
License(s): unknown
celeb-df-v2.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!pip install -q mtcnn opencv-python tqdm

import os, cv2, glob, random, shutil
import numpy as np
from mtcnn import MTCNN
from tqdm import tqdm

random.seed(42)

root = "/content/celebdf_v2/Celeb-DF-v2"
os.makedirs("data3/train/real", exist_ok=True)
os.makedirs("data3/train/fake", exist_ok=True)
os.makedirs("data3/val/real", exist_ok=True)
os.makedirs("data3/val/fake", exist_ok=True)

detector = MTCNN()

def extract_faces(video_path, out_dir, n_frames=1):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0: return
    idxs = np.linspace(0, total-1, n_frames, dtype=int)
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret: continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        faces = detector.detect_faces(rgb)
        if len(faces)==0: continue
        x,y,w,h = faces[0]['box']
        face = rgb[max(y,0):y+h, max(x,0):x+w]
        if face.size==0: continue
        face = cv2.resize(face, (224,224))
        fname = os.path.join(out_dir, f"{os.path.basename(video_path)}_{i}.jpg")
        cv2.imwrite(fname, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
    cap.release()

real_files = glob.glob(f"{root}/Celeb-real/*.mp4")
fake_files = glob.glob(f"{root}/Celeb-synthesis/*.mp4")

def split_and_process(files, label):
    random.shuffle(files)
    cut = int(len(files) * 0.8)
    train_files = files[:cut]
    val_files   = files[cut:]

    for v in tqdm(train_files, desc=f"{label} train"):
        extract_faces(v, f"data3/train/{label}")
    for v in tqdm(val_files, desc=f"{label} val"):
        extract_faces(v, f"data3/val/{label}")

split_and_process(real_files, "real")
split_and_process(fake_files, "fake")

print("✅ 전처리 완료")


fake val: 100%|██████████| 1128/1128 [05:13<00:00,  3.60it/s]

✅ 전처리 완료


In [ ]:
!pip install -q torch torchvision transformers tqdm pillow accelerate

import os
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPVisionModel
from PIL import Image
from tqdm import tqdm

device = "cuda"

class DeepfakeDataset(Dataset):
    def __init__(self, root, processor, train=False):
        self.samples = []
        for label, name in enumerate(["real","fake"]):  # ✅ real=0, fake=1
            folder = os.path.join(root, name)
            for f in os.listdir(folder):
                if f.lower().endswith((".jpg",".png")):
                    self.samples.append((os.path.join(folder,f), label))
        self.processor = processor
        self.train = train
        self.aug = T.Compose([
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2),
        ])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.train:
            img = self.aug(img)
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(), torch.tensor(label)

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

train_ds = DeepfakeDataset("data3/train", processor, train=True)
val_ds   = DeepfakeDataset("data3/val", processor, train=False)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32)

class DeepfakeCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch16")
        self.fc = nn.Sequential(
            nn.Linear(self.backbone.config.hidden_size,512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512,2)
        )
    def forward(self, x):
        x = self.backbone(pixel_values=x).pooler_output
        return self.fc(x)

model = DeepfakeCLIP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

for epoch in range(8):
    model.train()
    for x,y in tqdm(train_dl, desc=f"Epoch {epoch+1}"):
        x,y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()

model.eval()
correct = total = 0
with torch.no_grad():
    for x,y in val_dl:
        x,y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred==y).sum().item()
        total += y.size(0)

print(f"✅ Validation Accuracy: {100*correct/total:.2f}%")


from pathlib import Path

save_dir = "/content/drive/MyDrive/deepfake/model/CLIP-ViT-B16_1"
Path(save_dir).mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), f"{save_dir}/CLIP-ViT-B16_1.bin")
print(f"✅ Model saved to {save_dir}/CLIP-ViT-B16.bin")

Epoch 8: 100%|██████████| 164/164 [00:37<00:00,  4.39it/s]


✅ Validation Accuracy: 86.44%
✅ Model saved to /content/drive/MyDrive/deepfake/model/CLIP-ViT-B16_1/CLIP-ViT-B16.bin
